# Neo4j → Databricks: Enrich

**Run on dedicated compute. Pipeline step 3 of 3 (slide 12: "The Enrichment Pipeline"; slide 9: "Architecture at a Glance").**

Enrich, results land in Gold. The three GDS algorithm results now exist as node properties in Neo4j Aura. This notebook reads them back via the Spark Connector, joins with Silver, and writes three Delta tables that become the AFTER Genie space's catalog:

- **`gold_accounts`** (16 cols) — account metadata + `risk_score` + `community_id` + `similarity_score` + community aggregates (`community_size`, `community_avg_risk_score`, `community_risk_rank`, `inbound_transfer_events`) + derived flags (`is_ring_community`, `fraud_risk_tier`).
- **`gold_account_similarity_pairs`** (4 cols) — pairwise similarity scores with a `same_community` flag for intra- vs cross-community analysis.
- **`gold_fraud_ring_communities`** (8 cols) — one row per Louvain community, pre-aggregated with `member_count`, `avg_risk_score`, `avg_similarity_score`, `is_ring_candidate`, and `top_account_id`.

The pull direction matters: Neo4j does not write to Unity Catalog directly. Nothing in the graph reaches a Genie query except what this notebook materializes to Gold. The audit trail is the Delta table. Genie treats graph-derived columns as ordinary dimensions: `GROUP BY fraud_risk_tier`, `WHERE is_ring_candidate = true`.

**Canonical reference:** `enrichment-pipeline/jobs/03_pull_gold_tables.py` + `enrichment-pipeline/sql/gold_schema.sql` are the production source of truth. This workshop notebook mirrors that build with display() cells and inline narrative; keep the two in sync when thresholds or columns change.

## 0. Configuration

In [ ]:
CATALOG   = "graph-enriched-lakehouse"
SCHEMA    = "graph-enriched-schema"

NEO4J_URI      = dbutils.secrets.get("neo4j-graph-engineering", "uri")
NEO4J_USER     = dbutils.secrets.get("neo4j-graph-engineering", "username")
NEO4J_PASSWORD = dbutils.secrets.get("neo4j-graph-engineering", "password")

NEO4J_OPTS = {
    "url":                    NEO4J_URI,
    "authentication.basic.username": NEO4J_USER,
    "authentication.basic.password": NEO4J_PASSWORD,
}

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

In [ ]:
from pyspark.sql import functions as F

---
## 1. Read Graph Features from Neo4j

The Spark Connector reads the enriched Account nodes directly. All three
GDS-computed properties come along as columns.

In [ ]:
graph_features_df = (
    spark.read
    .format("org.neo4j.spark.DataSource")
    .options(**NEO4J_OPTS)
    .option("labels", "Account")
    .load()
    .select(
        F.col("account_id").cast("long"),
        F.col("risk_score").cast("double"),
        F.col("community_id").cast("long"),
        F.col("similarity_score").cast("double"),
    )
)

print(f"Read {graph_features_df.count()} Account nodes with graph features")
graph_features_df.show(10)

## 2. Write Graph Features to Delta

In [ ]:
FEATURES_TABLE     = f"`{CATALOG}`.`{SCHEMA}`.account_graph_features"
FEATURES_TABLE_API = f"{CATALOG}.{SCHEMA}.account_graph_features"

# Ensure account_id is NOT NULL so it can serve as a primary key
spark.sql(f"DROP TABLE IF EXISTS {FEATURES_TABLE}")

spark.sql(f"""
    CREATE TABLE {FEATURES_TABLE} (
        account_id     BIGINT       NOT NULL,
        risk_score     DOUBLE,
        community_id   BIGINT,
        similarity_score DOUBLE,
        CONSTRAINT account_graph_features_pk PRIMARY KEY (account_id)
    )
""")

graph_features_df.writeTo(FEATURES_TABLE).append()

print(f"Written to {FEATURES_TABLE} (with PK constraint)")

## 3. Register in Feature Store (Unity Catalog)

Makes graph features discoverable and joinable with any training dataset
by `account_id`.

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Read back from Delta (not the Neo4j source) to avoid Spark V2 push-down issues
graph_features_delta = spark.table(FEATURES_TABLE)

fe.write_table(
    name=FEATURES_TABLE_API,
    df=graph_features_delta,
    mode="merge",
)

print(f"Feature Store updated: {graph_features_delta.count()} rows")

---
## 4. Build the Full Training Table

Join original tabular features with graph-derived features to create
the complete training dataset.

In [ ]:
# Join accounts with account_labels to bring in is_fraud as the training label.
# The operational accounts table has no is_fraud column; labels live in account_labels.
accounts_df = (
    spark.table(f"`{CATALOG}`.`{SCHEMA}`.accounts")
    .join(spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_labels"), "account_id", "left")
)
graph_feat  = spark.table(FEATURES_TABLE)

# Tabular features from transactions
txn_features = (
    spark.table(f"`{CATALOG}`.`{SCHEMA}`.transactions")
    .groupBy("account_id")
    .agg(
        F.count("*").alias("txn_count"),
        F.round(F.avg("amount"), 2).alias("avg_txn_amount"),
        F.round(F.stddev("amount"), 2).alias("std_txn_amount"),
        F.round(F.max("amount"), 2).alias("max_txn_amount"),
        F.countDistinct("merchant_id").alias("unique_merchants"),
        F.round(F.avg("txn_hour").cast("double"), 2).alias("avg_txn_hour"),
        F.round(F.sum(F.when(F.col("txn_hour").between(0, 5), 1).otherwise(0)) / F.count("*"), 4)
            .alias("night_txn_ratio"),
    )
)

# P2P transfer features
p2p_features = (
    spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_links")
    .groupBy(F.col("src_account_id").alias("account_id"))
    .agg(
        F.count("*").alias("p2p_out_count"),
        F.round(F.avg("amount"), 2).alias("avg_p2p_amount"),
    )
)

# Full training table
training_df = (
    accounts_df
    .join(txn_features,  "account_id", "left")
    .join(p2p_features,  "account_id", "left")
    .join(graph_feat,    "account_id", "left")
    .fillna(0)
)

TRAINING_TABLE = f"`{CATALOG}`.`{SCHEMA}`.training_dataset"
(training_df
 .write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(TRAINING_TABLE)
)

print(f"Training table: {training_df.count()} rows, {len(training_df.columns)} columns")
training_df.printSchema()

## 5. Preview: Feature Correlations with Fraud

In [ ]:
display(
    training_df
    .groupBy("is_fraud")
    .agg(
        F.round(F.avg("risk_score"), 6).alias("avg_risk_score"),
        F.round(F.avg("similarity_score"), 4).alias("avg_similarity"),
        F.round(F.avg("txn_count"), 1).alias("avg_txn_count"),
        F.round(F.avg("unique_merchants"), 1).alias("avg_unique_merchants"),
        F.round(F.avg("night_txn_ratio"), 4).alias("avg_night_ratio"),
        F.round(F.avg("p2p_out_count"), 1).alias("avg_p2p_out"),
    )
)

---

### What's Next

Sections 1–5 built the `training_dataset` used by `06_train_model.ipynb`.

Sections 6–8 build the three Gold tables the AFTER Genie Space queries:

| Table | Columns | Role |
|-------|---------|------|
| `gold_accounts` | 16 | Account dimension with GDS features, community aggregates, and derived flags (`is_ring_community`, `fraud_risk_tier`) |
| `gold_account_similarity_pairs` | 4 | Pairwise similarity with a `same_community` flag |
| `gold_fraud_ring_communities` | 8 | One row per Louvain community, pre-aggregated with `is_ring_candidate` and `top_account_id` |

These sections are idempotent — re-running drops and recreates from the latest Neo4j state. The base Silver tables are never touched.

---
## 6. Build Gold Table 1: `gold_accounts`

Account dimension enriched with graph analytics features. Joins accounts with the three GDS features and computes community aggregates and derived flags:

- **`community_size` / `community_avg_risk_score` / `community_risk_rank`** — per-community window aggregates. `community_risk_rank=1` identifies the highest-similarity account in each community.
- **`inbound_transfer_events`** — count of `account_links` rows where the account is the destination.
- **`is_ring_community`** — true when the community has 50–200 members AND `community_avg_risk_score > 1.0`. These are the anomalous-size, high-centrality clusters the Louvain + PageRank combination surfaces.
- **`fraud_risk_tier`** — `high` for accounts in a ring community; `low` otherwise. This is what Genie filters and groups on.

Column comments are applied via `CREATE OR REPLACE TABLE` so Genie reads the descriptions. Thresholds mirror `enrichment-pipeline/jobs/_gold_constants.py`; keep in sync when they change.

**`is_fraud` is intentionally excluded** so Genie has to work from graph structure, not from the explicit label.

In [ ]:
from pyspark.sql import Window

# Thresholds mirror enrichment-pipeline/jobs/_gold_constants.py. Changing them here
# without updating that file will cause the workshop and production builds
# to drift.
RING_SIZE_LOW = 50
RING_SIZE_HIGH = 200
COMMUNITY_AVG_RISK_MIN = 1.0

GOLD_ACCOUNTS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.gold_accounts"

# Schema with Unity Catalog column comments — Genie reads descriptions to
# understand what each column means. Mirrors enrichment-pipeline/sql/gold_schema.sql.
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_ACCOUNTS_TABLE} (
    account_id               BIGINT   NOT NULL COMMENT 'Account identifier (joins to accounts.account_id)',
    account_hash             STRING   COMMENT 'Anonymized account identifier',
    account_type             STRING   COMMENT 'Account category: checking, savings, or business',
    region                   STRING   COMMENT 'Geographic region where the account was opened',
    balance                  DOUBLE   COMMENT 'Current account balance in USD',
    opened_date              DATE     COMMENT 'Date the account was opened',
    holder_age               INT      COMMENT 'Age of the account holder in years',
    risk_score               DOUBLE   COMMENT 'PageRank centrality score on the account-to-account transfer graph. Measures how influential an account is in the transfer network. Null for accounts below the minimum degree threshold.',
    community_id             BIGINT   COMMENT 'Louvain community label. Accounts that predominantly transfer money among themselves share the same community_id. Null for accounts below the minimum degree threshold.',
    similarity_score         DOUBLE   COMMENT 'Jaccard similarity score from the shared-merchant bipartite graph. Measures overlap in merchant visit patterns with other accounts in the same community. Null for accounts below the minimum degree threshold.',
    community_size           BIGINT   COMMENT 'Number of accounts sharing this community_id',
    community_avg_risk_score DOUBLE   COMMENT 'Mean risk_score across all accounts in the community',
    community_risk_rank      INT      COMMENT 'Rank of this account within its community, ordered by similarity_score descending, then risk_score descending, then account_id ascending. Rank 1 = the account with the highest merchant-overlap similarity in the community.',
    inbound_transfer_events  BIGINT   COMMENT 'Count of account_links rows where this account is the transfer destination (dst_account_id)',
    is_ring_community        BOOLEAN  COMMENT 'True when the account community has between 50 and 200 members and a community_avg_risk_score above 1.0, indicating a tightly-knit transfer cluster of anomalous size and centrality',
    fraud_risk_tier          STRING   COMMENT 'Pre-computed binary risk classification based on community membership. Values: high (is_ring_community=true), low (all other accounts).'
)
USING DELTA
COMMENT 'Account dimension enriched with graph analytics features derived from the transfer network'
""")

# Inbound transfer count per destination account
inbound_counts = (
    spark.table(f"`{CATALOG}`.`{SCHEMA}`.account_links")
    .groupBy(F.col("dst_account_id").alias("account_id"))
    .agg(F.count("*").alias("inbound_transfer_events"))
)

# Per-community windows. row_number (not rank) with an account_id tiebreak
# so community_risk_rank=1 is exactly one row per community.
w_community = Window.partitionBy("community_id")
w_community_rank = Window.partitionBy("community_id").orderBy(
    F.desc("similarity_score"), F.desc("risk_score"), F.asc("account_id")
)

# graph_feat is the Delta copy of Neo4j's Account GDS features written in section 2.
graph_feat = spark.table(FEATURES_TABLE)

# Unscored accounts keep null community_id/risk_score/similarity_score — a
# blanket fillna(0) would bucket them all into a synthetic community_id=0
# and poison the window aggregates. inbound_transfer_events is zero-filled
# because missing means "no inbound transfers," not "unknown."
gold_df = (
    spark.table(f"`{CATALOG}`.`{SCHEMA}`.accounts")
    .join(graph_feat, "account_id", "left")
    .join(inbound_counts, "account_id", "left")
    .fillna({"inbound_transfer_events": 0})
    .withColumn("community_size", F.count("*").over(w_community))
    .withColumn("community_avg_risk_score", F.avg("risk_score").over(w_community))
    .withColumn("community_risk_rank", F.row_number().over(w_community_rank))
    .withColumn(
        "is_ring_community",
        F.col("community_size").between(RING_SIZE_LOW, RING_SIZE_HIGH)
        & (F.col("community_avg_risk_score") > COMMUNITY_AVG_RISK_MIN),
    )
    .withColumn(
        "fraud_risk_tier",
        F.when(F.col("is_ring_community"), "high").otherwise("low"),
    )
    .select(
        "account_id", "account_hash", "account_type", "region",
        "balance", "opened_date", "holder_age",
        "risk_score", "community_id", "similarity_score",
        "community_size", "community_avg_risk_score", "community_risk_rank",
        "inbound_transfer_events", "is_ring_community", "fraud_risk_tier",
    )
    .cache()
)

n_gold = gold_df.count()  # materializes the cache; reused in sections 7 and 8

(gold_df
 .write.format("delta").mode("overwrite")
 .option("overwriteSchema", "false")
 .saveAsTable(GOLD_ACCOUNTS_TABLE)
)

print(f"Written {GOLD_ACCOUNTS_TABLE} ({n_gold:,} rows, 16 columns)")
display(gold_df.orderBy(F.desc("risk_score")).limit(10))

---
## 7. Build Gold Table 2: `gold_account_similarity_pairs`

Pairs of accounts connected by a `SIMILAR_TO` relationship in Neo4j, one row per unique pair. The `same_community` flag lets Genie compare intra- vs cross-community similarity: ring members that share merchants typically also share a Louvain community, so `same_community=true` marks pairs worth deeper investigation.

Pair ordering is deterministic — `account_id_a` is always the smaller ID, `account_id_b` the larger. This removes duplicate `(a,b)` / `(b,a)` rows.

In [ ]:
GOLD_PAIRS_TABLE = f"`{CATALOG}`.`{SCHEMA}`.gold_account_similarity_pairs"

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_PAIRS_TABLE} (
    account_id_a     BIGINT   COMMENT 'First account in the pair, always the smaller account_id (joins to gold_accounts.account_id)',
    account_id_b     BIGINT   COMMENT 'Second account in the pair, always the larger account_id (joins to gold_accounts.account_id)',
    similarity_score DOUBLE   COMMENT 'Jaccard similarity score based on shared merchant visits between account_id_a and account_id_b',
    same_community   BOOLEAN  COMMENT 'True when account_id_a and account_id_b share the same non-null community_id in gold_accounts'
)
USING DELTA
COMMENT 'Account pairs connected by a shared-merchant similarity edge — one row per unique pair'
""")

# Pull SIMILAR_TO edges from Neo4j via the Spark Connector
similarity_pairs_df = (
    spark.read
    .format("org.neo4j.spark.DataSource")
    .options(**NEO4J_OPTS)
    .option("relationship", "SIMILAR_TO")
    .option("relationship.source.labels", ":Account")
    .option("relationship.target.labels", ":Account")
    .load()
    .select(
        F.least(F.col("`source.account_id`"), F.col("`target.account_id`"))
            .cast("long").alias("account_id_a"),
        F.greatest(F.col("`source.account_id`"), F.col("`target.account_id`"))
            .cast("long").alias("account_id_b"),
        F.col("`rel.similarity_score`").cast("double").alias("similarity_score"),
    )
    .dropDuplicates(["account_id_a", "account_id_b"])
)

# Add same_community via gold_df (cached from section 6). Guard both sides
# non-null before equality so pairs involving an unscored account come out
# as false, not null.
community_lookup = gold_df.select("account_id", "community_id")

similarity_pairs_df = (
    similarity_pairs_df
    .join(
        community_lookup
            .withColumnRenamed("account_id", "account_id_a")
            .withColumnRenamed("community_id", "community_id_a"),
        "account_id_a", "left",
    )
    .join(
        community_lookup
            .withColumnRenamed("account_id", "account_id_b")
            .withColumnRenamed("community_id", "community_id_b"),
        "account_id_b", "left",
    )
    .withColumn(
        "same_community",
        F.col("community_id_a").isNotNull()
        & F.col("community_id_b").isNotNull()
        & (F.col("community_id_a") == F.col("community_id_b")),
    )
    .drop("community_id_a", "community_id_b")
    .cache()
)

n_pairs = similarity_pairs_df.count()

(similarity_pairs_df
 .write.format("delta").mode("overwrite")
 .option("overwriteSchema", "false")
 .saveAsTable(GOLD_PAIRS_TABLE)
)

print(f"Written {GOLD_PAIRS_TABLE} ({n_pairs:,} rows)")
display(similarity_pairs_df.orderBy(F.desc("similarity_score")).limit(10))

---
## 8. Build Gold Table 3: `gold_fraud_ring_communities`

One row per Louvain community, pre-aggregated for ring-level analysis. Lets Genie answer questions like "which ring-candidate communities have the most members?" without re-aggregating `gold_accounts` at query time.

- **`is_ring_candidate`** — true when the community has 50–200 members AND `avg_risk_score > 1.0`. Same predicate as `gold_accounts.is_ring_community`, but here it sits on the per-community row. The AFTER Genie questions use `is_ring_candidate = true` as the ring-candidate filter.
- **`top_account_id`** — the single account with the highest similarity in each community (ties broken by `risk_score` descending, then `account_id`). Sorting on similarity first (not risk) prevents a high-PageRank "whale" that coincidentally lands in a ring community from being picked over actual ring members.

In [ ]:
GOLD_RING_COMMUNITIES_TABLE = f"`{CATALOG}`.`{SCHEMA}`.gold_fraud_ring_communities"

spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_RING_COMMUNITIES_TABLE} (
    community_id           BIGINT   COMMENT 'Community identifier (joins to gold_accounts.community_id)',
    member_count           BIGINT   COMMENT 'Number of accounts in this community',
    avg_risk_score         DOUBLE   COMMENT 'Mean graph centrality score across all community members',
    max_risk_score         DOUBLE   COMMENT 'Highest graph centrality score within the community',
    avg_similarity_score   DOUBLE   COMMENT 'Mean merchant-visit similarity score across community members',
    high_risk_member_count BIGINT   COMMENT 'Number of accounts in this community with risk_score above 1.0',
    is_ring_candidate      BOOLEAN  COMMENT 'True when member_count is between 50 and 200 and avg_risk_score is above 1.0. These communities show anomalous size and centrality consistent with tight transfer rings.',
    top_account_id         BIGINT   COMMENT 'The account with the highest similarity_score in this community, ties broken by risk_score descending then account_id ascending. Identifies the most structurally similar account rather than the most transfer-central one.'
)
USING DELTA
COMMENT 'Louvain community summary — one row per community, pre-aggregated for ring-level analysis'
""")

# Community-level aggregates over gold_df (cached from section 6)
ring_aggregates = (
    gold_df
    .filter(F.col("community_id").isNotNull())
    .groupBy("community_id")
    .agg(
        F.count("*").alias("member_count"),
        F.round(F.avg("risk_score"), 6).alias("avg_risk_score"),
        F.round(F.max("risk_score"), 6).alias("max_risk_score"),
        F.round(F.avg("similarity_score"), 5).alias("avg_similarity_score"),
        F.sum(F.when(F.col("risk_score") > 1.0, 1).otherwise(0))
            .alias("high_risk_member_count"),
    )
    .withColumn(
        "is_ring_candidate",
        F.col("member_count").between(RING_SIZE_LOW, RING_SIZE_HIGH)
        & (F.col("avg_risk_score") > COMMUNITY_AVG_RISK_MIN),
    )
)

# top_account_id: highest similarity account per community, tiebreak by
# risk_score then account_id. Matches gold_accounts.community_risk_rank=1.
w_top = Window.partitionBy("community_id").orderBy(
    F.desc("similarity_score"), F.desc("risk_score"), F.asc("account_id")
)

top_accounts = (
    gold_df
    .filter(F.col("community_id").isNotNull())
    .select("community_id", "account_id", "risk_score", "similarity_score")
    .withColumn("_row", F.row_number().over(w_top))
    .filter(F.col("_row") == 1)
    .select(F.col("community_id"), F.col("account_id").alias("top_account_id"))
)

ring_communities_df = ring_aggregates.join(top_accounts, "community_id", "left").cache()

counts = ring_communities_df.agg(
    F.count("*").alias("total"),
    F.sum(F.col("is_ring_candidate").cast("int")).alias("candidates"),
).collect()[0]

(ring_communities_df
 .write.format("delta").mode("overwrite")
 .option("overwriteSchema", "false")
 .saveAsTable(GOLD_RING_COMMUNITIES_TABLE)
)

print(
    f"Written {GOLD_RING_COMMUNITIES_TABLE} "
    f"({int(counts['total']):,} rows, {int(counts['candidates'] or 0)} ring candidates)"
)
display(
    ring_communities_df
    .filter(F.col("is_ring_candidate"))
    .orderBy(F.desc("avg_risk_score"))
    .limit(15)
)

---
## Done

The three Gold tables are written.

| Table | Role in the AFTER Genie Space |
|-------|-------------------------------|
| `gold_accounts` | Filter on `fraud_risk_tier='high'`, group by `community_id`, join against Silver on `account_id`. |
| `gold_account_similarity_pairs` | Count or average pair-level similarity within/across communities via `same_community`. |
| `gold_fraud_ring_communities` | Filter on `is_ring_candidate=true` for the ring-candidate cohort; join on `top_account_id` for a representative account per community. |

**Next →** The AFTER Genie Space now has the columns it needs. Re-run the AFTER cells in `01_genie_silver_questions.ipynb` to see the before/after gap live, or use `genie-guide.md` for copy-paste questions in Genie directly.

For ML-focused Q&A, continue to `06_train_model.ipynb` (supplementary, off the 15-minute demo path).